# CellQuery — Data Exploration & Attention Visualization

Explore GDSC2 data and visualize cross-attention weights from the trained model.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import torch

from src.config import load_config, resolve_device
from src.data.dataset import get_dataloaders
from src.models.model import CellDrugModel

cfg = load_config(ROOT / "config" / "default.yaml")
device = resolve_device(cfg.training.device)
print(f"Device: {device}")

In [ ]:
train_loader, valid_loader, test_loader, meta = get_dataloaders(cfg)
batch = next(iter(train_loader))
print("Cell features:", batch["cell_features"].shape)
print("Labels (first 5):", batch["label"][:5])
print("Graph nodes:", batch["graph"].x.shape)

In [ ]:
model = CellDrugModel.from_config(cell_dim=meta["cell_dim"], cfg=cfg).to(device)
model.eval()

with torch.no_grad():
    graph = batch["graph"].to(device)
    cell = batch["cell_features"].to(device)
    out = model(graph, cell, return_attention=True)

pred = out["prediction"].cpu().numpy()
true = batch["label"].numpy()
for p, t in zip(pred[:5], true[:5]):
    print(f"pred={p:.3f}  true={t:.3f}")

In [ ]:
import numpy as np

attn_layer0 = out["attention_weights"][0][0]
if attn_layer0.numel() > 0:
    weights = attn_layer0.cpu().numpy()
    plt.figure(figsize=(10, 4))
    plt.imshow(weights, aspect="auto", cmap="viridis")
    plt.colorbar(label="attention")
    plt.xlabel("atom index")
    plt.ylabel("probe index")
    plt.title("Layer 0: cell probes -> drug atoms")
    plt.tight_layout()
    plt.show()
else:
    print("No attention weights to display.")